In [1]:
import os
import pandas as pd
import numpy as np
from collections import defaultdict, Counter
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
import seaborn as sb
import csv
from scipy import stats
from scipy.spatial.distance import cosine
from nltk.tokenize import TweetTokenizer
from nltk.corpus import stopwords
from nltk.sentiment.vader import SentimentIntensityAnalyzer

In [2]:
post_info = pd.read_csv("perspective_toxicity.csv")

post_info['ih_austin_a'] = np.mean([post_info.CURIOSITY_EXPERIMENTAL,
                                    post_info.NUANCE_EXPERIMENTAL,
                                    post_info.RESPECT_EXPERIMENTAL], axis=0)

post_info['ih_austin_b'] = np.mean([post_info.CURIOSITY_EXPERIMENTAL,
                                    post_info.NUANCE_EXPERIMENTAL,
                                    post_info.REASONING_EXPERIMENTAL,
                                    post_info.RESPECT_EXPERIMENTAL], axis=0)

post_info['tox_austin_a'] = post_info.TOXICITY_EXPERIMENTAL

post_info['tox_austin_b'] = np.mean([post_info.TOXICITY_EXPERIMENTAL,
                                     post_info.IDENTITY_ATTACK_EXPERIMENTAL,
                                     post_info.INSULT_EXPERIMENTAL,
                                     post_info.THREAT_EXPERIMENTAL,
                                     post_info.SEXUALLY_EXPLICIT], axis=0)



In [3]:
for x in ['CURIOSITY_EXPERIMENTAL','NUANCE_EXPERIMENTAL','REASONING_EXPERIMENTAL','RESPECT_EXPERIMENTAL',
          'ih_austin_a','ih_austin_b']:
    posts = post_info.sort_values(x, ascending=False).head(10)
    print(x)
    for ind, row in posts.iterrows():
        print(">> " + row['postOrCommentText'])
    print()

CURIOSITY_EXPERIMENTAL
>> What is your objective criteria for identifying those people though? I agree wholeheartedly, but how can you stop someone for lying or hiding their issues without tighter background checks?
>> Agree completely. What is your take on the mass migrations into the country through our Southern border?
>> Hmmm how do we think outbursts and crimes of passion play into this from people who wouldn’t typically be considered violent? You can’t police everyone but wouldn’t stricter gun laws help limit deaths due to gun violence over all?
>> Could you further elaborate on why you feel there should be gun bans?
>> What exactly is reasonable gun legislation? What would that look like? What exactly would it entail?
>> I agree, but wouldn't redtricts like background checks and no sales to domestic violence convicted persons be ok?
>> What would those special circumstances look like though? Who is allowed to have access to these guns and who is not?
>> I'd be interested in your

In [4]:
for x in ["TOXICITY_EXPERIMENTAL", "IDENTITY_ATTACK_EXPERIMENTAL", "INSULT_EXPERIMENTAL",
          "THREAT_EXPERIMENTAL", "SEXUALLY_EXPLICIT", "tox_austin_b"]:
    posts = post_info.sort_values(x, ascending=False).head(10)
    print(x)
    for ind, row in posts.iterrows():
        print(">> " + row['postOrCommentText'])
    print()

TOXICITY_EXPERIMENTAL
>> i think u should suck a dick
>> Clueless fucking liberals don't get it
>> Way to paint brush and fucking label people 
>> A border wall is the stupidest idea from a traitor scamming orange person with fake hair named Donold 
>> You are a fool if you think it will stop it.  Look at drugs, hows that fight going?
>> Dumb  sooooo dumb
>> . Any idiot can use a military style weapon to spray bullets uncontrolled and do serious damage. While a regular gun can still cause harm, I’d argue it would require more skill to devastate a mass population. There would also be a greater chance of stopping a lone gunmen.
>> I think the government needs to pull their heads out of their backside orifices and get a better grip on the southern border…. Obvious what they are doing now isn’t working.  And no fn wall…. Thats a waste of money and looks stupid af.
>> American citizens are dumb for thinking their guns would be enough to stop an authoritarian government
>> Guns don't kill pe

In [5]:
post_info[['ih_austin_a','ih_austin_b','tox_austin_a','tox_austin_b']].corr()

,ih_austin_a,ih_austin_b,tox_austin_a,tox_austin_b
ih_austin_a,1.000000,0.934668,-0.317300,-0.284845
ih_austin_b,0.934668,1.000000,-0.251984,-0.228471
tox_austin_a,-0.317300,-0.251984,1.000000,0.978241
tox_austin_b,-0.284845,-0.228471,0.978241,1.000000


# IH Post Setup

In [6]:
ih_posts = ["A lot of reporting I've seen suggests that the illegal crossing are very dangerous. If that's true, we should do everything we can to deter this. I've heard that many die every year crossing in unsafe areas, and that even when they make it across they die of heat and thrust or crammed in the back of trucks. It seems like, then, a border wall is for their safety as well as ours.",
            "I get the reason some people want gun control.  I want them to understand that where I live, we need guns. We are over run with coyotes. They kill our pets. We try and kill them. You have to use a weapon with range. What youd call an assault weapon is perfect for that, and they are popular here for coyotes. I just scare them with my gun, because I don't ever hit them",
            "I might be wrong but it seems like the places that have the most mayhem are those with the most restrictive laws. I think about the pressure cooker bombs a few years back at the marathon.",
            "I totally understand, it's not the guns fault. We shouldn't do away with guns because of recent incidents, but it seems to me that we need to look at those who use the guns. I think regulating who can get access to them could help solve this issue. I agree that people have the need to feel safe and they should secure their environment the way they feel comfortable",
            "I try not to be naive that utopia exists or can exist. I know some people think differently, but I do believe in laws and rules and that people are basically good and follow them most of the time. I'd love to see data on this, but I feel like it's only the fringe that does not follow them and causes mayhem for the rest of us.",
            "I understand it is a complex issue and I actually have no problem with asylum seekers. I personally believe that we should be letting in people who will come in to this country the legal way and assimilate into our country. This is just my perspective and open to hear the views of others.",
            "I understand why people are worried about the Southern Border situation, but I don't think that has much to do with terrorism. I might be wrong, but I think that more terrorists are coming over the Canadian border.",
            "I understand why some people want  gun control.  It would help insure guns aren't  given to unstable people. But I also believe that as an American I have every right to defend my property, family, and myself with a gun or guns. I worry only criminals would have weapons.",
            "I'm honestly curious about what the other side thinks. I agree that the gangs and the drugs that come over here from Mexico are a problem. At the same time,  I think we already have gangs and drugs over here before Mexico brought them over.",
            "I'm no psychologist, but I don't think that criminals care what the laws are and what they can or can't do. If that is true, they will obtain a gun by whatever means they can, carry it illegally, use it in a crime and possibly use it to rob you.",
            "I'm not an expert on gun policy, but I did write research papers on the topic in college. While doing research, I was persuaded there was a lot misinformation put out by the NRA, which wa led by someone personally enriching himself from the sales of ammunition.",
            "I'm open to hearing other views, but it sure seems like gun control doesn't work. I've heard that the states with the strictest gun control has the highest crime rates and murder rates. I would think that's because only criminal have the guns.",
            "I'm sure others disagree, but I don't think it's our responsibility as a nation to take care of people from other countries who can't get any help from their own government. I think we have enough AMERICANS who need assistance, and we should help them first. I guess I just don't understand why we should be paying for anyone who manages to sneak in to our country.",
            "It's been a long time since I took US history, but I believe this land as far as Texas, New Mexico, Arizona, California, we're all Mexico territory before The United States took it from them. So I guess it would make sense that They're going to come back, maybe because they never left.",
            "Living close to the border has made me think that we need a wall. Our border patrol, who are the real experts, has asked for a wall. I haven't dug into the numbers, but I've heard the wall has already made in a difference in the areas it is in place. I feel we shouldn't ask our border patrol to risk their lives and not give them the tools they ask for.",
            "Others might disagree, but I don't understand the argument that  Mexicans are taking jobs.  If you are unemployed, couldn't find a job anywhere, and a lettuce field in your city that was hiring, would you apply? My sense is that most Americans would say no.",
            "Others might disagree, but I think we need reasonable and uniform NATIONAL gun legislation. I think one reason for the failure of gun control laws in the US is the pathwork of state-to-state differences. I might be wrong, but I think Congress even forbid the government trackingof gun violence deaths.",
            "I'm not an expert, but it seems like more immigration can only hurt me. I'm interesting in hearing the argument for the other side.",
            "I understand the argument that the system needs reform. However, my own view is that many who have immigrated are the epitome of an outstanding citizens that contributes to society and our economy.",
            "I'm open to other ideas, but I wonder if we should require every gun sale to be registered and require gun manufacturers to include trigger locks at no additional charge to the consumer.  It probably won't solve all our problems, but I think that we can take measures to reduce gun deaths without trampling on the 2nd Amendment."]



In [7]:
all_posts = pd.read_csv("participant_content.csv")
all_posts = all_posts[all_posts.userContent == 0]

In [8]:
ih = []

for p in ih_posts:
    ids = list(all_posts[all_posts.postOrCommentText.apply(lambda x: x == p)].postOrCommentId)
    if len(ids) == 0:
        print(p)
    ih += ids

ih

['654e2eccf18dbd001f7ab0a8',
 '655d0ae8f1c1b5001f5cb59c',
 '655d0ae8f1c1b5001f5cb55f',
 '654e2eccf18dbd001f7ab0a8',
 '655d0ae8f1c1b5001f5cb55f',
 '654e2eccf18dbd001f7ab0a8',
 '654e2eccf18dbd001f7ab0a8',
 '655d0ae8f1c1b5001f5cb59c',
 '654e2eccf18dbd001f7ab0a8',
 '654e2eccf18dbd001f7ab0a8',
 '655d0ae8f1c1b5001f5cb59c',
 '655d0ae8f1c1b5001f5cb59c',
 '655d0ae8f1c1b5001f5cb59c',
 '655d0ae8f1c1b5001f5cb59c',
 '654e2eccf18dbd001f7ab0a8',
 '655d0ae8f1c1b5001f5cb55f',
 '654e2eccf18dbd001f7ab0a8',
 '654e2eccf18dbd001f7ab0a8',
 '654e2eccf18dbd001f7ab0a8',
 '655d0ae8f1c1b5001f5cb55f',
 '654e2eccf18dbd001f7ab0a8',
 '655d0ae8f1c1b5001f5cb59c',
 '654e2eccf18dbd001f7ab0a8',
 '654e2eccf18dbd001f7ab0a8',
 '655d0ae8f1c1b5001f5cb55f',
 '654e2eccf18dbd001f7ab0a8',
 '655d0ae8f1c1b5001f5cb55f',
 '655d0ae8f1c1b5001f5cb55f',
 '655d0ae8f1c1b5001f5cb59c',
 '655d0ae8f1c1b5001f5cb55f',
 '654e2eccf18dbd001f7ab0a8',
 '655d0ae8f1c1b5001f5cb59c',
 '655d0ae8f1c1b5001f5cb55f',
 '654e2eccf18dbd001f7ab0a8',
 '654e2eccf18d

# Lib/Con Post Setup

In [9]:
lib_posts = [
"I understand why people are worried about the Southern Border situation, but I don't think that has much to do with terrorism. I might be wrong, but I think that more terrorists are coming over the Canadian border.",
"Others might disagree, but I don't understand the argument that  Mexicans are taking jobs.  If you are unemployed, couldn't find a job anywhere, and a lettuce field in your city that was hiring, would you apply? My sense is that most Americans would say no.",
"It's been a long time since I took US history, but I believe this land as far as Texas, New Mexico, Arizona, California, we're all Mexico territory before The United States took it from them. So I guess it would make sense that They're going to come back, maybe because they never left.",
"Others might disagree, but I think we need reasonable and uniform NATIONAL gun legislation. I think one reason for the failure of gun control laws in the US is the pathwork of state-to-state differences. I might be wrong, but I think Congress even forbid the government trackingof gun violence deaths.",
"I'm open to other ideas, but I wonder if we should require every gun sale to be registered and require gun manufacturers to include trigger locks at no additional charge to the consumer.  It probably won't solve all our problems, but I think that we can take measures to reduce gun deaths without trampling on the 2nd Amendment.",
"The problem is not guns; it's the people who have them. I know lots of people that I'd be comfortable having any type of gun. We need to restrict gun possession to responsible people - not drunks, or wife-beaters, not white supremesists. And not political extremeidts. Pretty obvious.",
"Background checks will stop some people who should not own guns from owning them and that's a start. We have finance laws we know people will break but we still do what we can to try to police it.",
"We need mandatory training, licensing, and spaces where guns are and are not allowed.",
"We live in a society and pay taxes and agree to social norms for protection. We do not live in the world as independent, armed encampments. It may cost me or a loved one our lives, but, living gun-free is the price for a civilized society.",
"We need stricter gun laws for modern times. At this point people don't need the right to bare arms. As a civilization, we've moved past this primal state. Our laws need a reality makeover.",
"I say yes to gun control. We must stand up to the NRA!",
"Stricter gun laws would help in many situations. It would definitely help decrease the incidence of gun violence. No one is saying that people will not be able to buy/own guns anymore. The government is not taking away your right to bear arms, it would just make it harder for those who may not be fit to own a gun to buy one.",
"Military weapons should not be sold to the public. No one that isn't in the military needs an AK-47 or similar weapon. These types of weapons were specifically created for war and not for public use or entertainment.",
"There should be gun bans even though gun enthusiasts will riot in response.",
"A good guy with a gun can kill a bad guy with a gun, but what if there were no guns at all? Protection and security should be left to law enforcement.",
"Automatic rifles need to be illegal.",
"If gun control means gun ban, then yes. Though I'm all for registration.",
"I am for gun control: ban them in the US unless there are special circumstances to own them.",
"I think gun control will stop all the mass shootings.",
"Guns are dangerous",
"The bottom line is that we benefit from the human capital that comes here legally to grow our economy, to innovate and lend their skills and talent to a country that welcomes them. Tech and finance would absolutely shut down without the H1-B visa program. On the other end of the spectrum, I see undocumented workers cleaning gutters all the time.",
"A wall is an expensive an useless symbolic endeavour that won't actually do anything to stop or slow immigration. 90% of people in this country illegally entered with valid work or tourism visas and then just did not leave when they were supposed to. It's the clogged up immigration court system that leads to undocumented immigrants more than an unsecured border.",
"Illegal immigrants do pay taxes. There is a tax id number specifically for illegal immigrants. Everyone pays sales tax at checkout for groceries etc. Unless you are homeless you pay rent or own a home. Part of that is paying school and property taxes.",
"Many farms in the past years have had significant waste due to a shortage of labor not being able to harvest. This increases food prices. The largest shortage of labor in healthcare is in nursing home jobs and home health aids. Immigrants frequently fill these jobs.",
"My view is that benefits of immigration outweigh the harm, by a lot. We gain smart and able folks. My doctor for the past 10 years was an immigrant, and I really liked the care he gave me. Plus immigrants will sometimes take jobs native born don't want,like house cleaning.",
"Immigrants seem to do more to try and support themselves than many native-born Americans",
"Historically, America is a nation of immigrants. And each group has contributed to the development of our society. Politicians just play on the fears of the working class to build opposition to immigration.",
"In my view, first-generation immigrants are the hardest-working, family-oriented, caring and driven people. Anyone with a heart would be welcoming them into your community. They are not the enemies, whatever their immigration status. They exhibit the values our country holds dear.",
"We need a way to get the labor we need, without exploitation, build community by decriminalizing immigration, and recognize that our country's actions in the world have done a great deal to create the conditions many people who come here are fleeing.",
"Immigration is the reason several countries, such as the United States, exist. Immigrants bring their culture, knowledge, and abilities to the new countries they inhabit.",
"Immigration has always been an integral part of the policy in the USA and should definatly continue.",
"I'm honestly curious about what the other side thinks. I agree that the gangs and the drugs that come over here from Mexico are a problem. At the same time,  I think we already have gangs and drugs over here before Mexico brought them over.",
"I understand the argument that the system needs reform. However, my own view is that many who have immigrated are the epitome of an outstanding citizens that contributes to society and our economy.",
"The US needs immigration in order to meet the economic needs of this country.",
"I believe the benefits of legal immigration outweigh the downsides",
"I'm not an expert on gun policy, but I did write research papers on the topic in college. While doing research, I was persuaded there was a lot misinformation put out by the NRA, which wa led by someone personally enriching himself from the sales of ammunition.",
"I feel that there are certain people that are mentally incapable of handling a gun. There should be some system that helps sift the mentally ill out from those who are capable and able to use a gun appropriately. I'm not sure about the specifics, though...",
"I totally understand, it's not the guns fault. We shouldn't do away with guns because of recent incidents, but it seems to me that we need to look at those who use the guns. I think regulating who can get access to them could help solve this issue. I agree that people have the need to feel safe and they should secure their environment the way they feel comfortable",
]

con_posts = [
"I'm not an expert, but it seems like more immigration can only hurt me. I'm interesting in hearing the argument for the other side.",
"I'm sure others disagree, but I don't think it's our responsibility as a nation to take care of people from other countries who can't get any help from their own government. I think we have enough AMERICANS who need assistance, and we should help them first. I guess I just don't understand why we should be paying for anyone who manages to sneak in to our country.",
"I understand it is a complex issue and I actually have no problem with asylum seekers. I personally believe that we should be letting in people who will come in to this country the legal way and assimilate into our country. This is just my perspective and open to hear the views of others.",
"Living close to the border has made me think that we need a wall. Our border patrol, who are the real experts, has asked for a wall. I haven't dug into the numbers, but I've heard the wall has already made in a difference in the areas it is in place. I feel we shouldn't ask our border patrol to risk their lives and not give them the tools they ask for.",
"A lot of reporting I've seen suggests that the illegal crossing are very dangerous. If that's true, we should do everything we can to deter this. I've heard that many die every year crossing in unsafe areas, and that even when they make it across they die of heat and thrust or crammed in the back of trucks. It seems like, then, a border wall is for their safety as well as ours.",
"Why does it have to be immigrants who forever change the cultural fabric of our country? We should preserve American identity & culture rather than make it a hodgepodge of different sects of world cultures with no overarching identity or commonality.",
"While you can name a handful of brilliant, successful immigrants, they aren't representative of the 1 million we take in each year. We should drastically reduce that number if not decrease it to net zero.",
"We need the border wall. As a country, we need to be able to secure our borders, and I have felt that way for a long time.",
"Our current lack of control over our own borders and outdated immigration laws have not served us well lately (nor even served the immigrants themselves well). We need to update our immigration laws to be more selective, enforce them more consistently, and remove all the illegals within our borders.",
"Illegals clog up the welfare office, free clinics, YMCA, ect. Without paying any taxes that support these things. Also they tend to have children soon as they get here making it impossible to deport them as the child is concidered an amercian.",
"We should require immigrants tto meet the same ideals we had in the 50's. My parents had to learn the language, pass a test and have a reason to be here and for my Dad that was a job that required education. The biggest problem I think we have is the sheer numbers of undocumented immigrants.",
"The immigration needs to be controlled in order to maintain any semblance of national identity. Past immigrants came to be a part of the great United States, to become part of us. They would embrace our culture and values and contribute theirs to make us all better. If the influx of immigrants is uncontrolled we run the risk of losing that unified goal.",
"Immigrants just come to get what they can, to take the opportunities living here affords without really investing energy and emotion into becoming an American. This further promotes divisions among us as different cultures just gather in various areas, re-establishing pockets of their own culture, rather than assimilating into ours",
"Immigration was good when they came and assimilated to the country's core values",
"Immigration is dangerous",
"I think its crazy to insist teachers learn spanish to teach illegals",
"I don't support immigration if people are coming here illegally",
"Crime rates are lower among illegals then citizens because there are more citizens then illegals.",
"I might be wrong but it seems like the places that have the most mayhem are those with the most restrictive laws. I think about the pressure cooker bombs a few years back at the marathon.",
"I try not to be naive that utopia exists or can exist. I know some people think differently, but I do believe in laws and rules and that people are basically good and follow them most of the time. I'd love to see data on this, but I feel like it's only the fringe that does not follow them and causes mayhem for the rest of us.",
"I'm no psychologist, but I don't think that criminals care what the laws are and what they can or can't do. If that is true, they will obtain a gun by whatever means they can, carry it illegally, use it in a crime and possibly use it to rob you.",
"I understand why some people want  gun control.  It would help insure guns aren't  given to unstable people. But I also believe that as an American I have every right to defend my property, family, and myself with a gun or guns. I worry only criminals would have weapons.",
"I'm open to hearing other views, but it sure seems like gun control doesn't work. I've heard that the states with the strictest gun control has the highest crime rates and murder rates. I would think that's because only criminal have the guns.",
'''Folks say ""those who are a threat"" shouldn't have guns. Felons -- even white-collar, non-violent felons who have done their time, are denied forever the right to own or possess a gun. I don't think we should strip those rights forever.''',
"It's impossible to control illegal gun sales, just like illegal drugs on the streets. If the bad people want them, they will get them. But the people who typically get them for good intentions are to protect themselves against the bad people with bad intentions so it just comes full circle. Guns should be for protections against the bad but now the bad includes people with guns.",
"Gun control will not stop criminals from getting black market guns.",
"Not one mass murderer was an NRA member. Why is that? Did you know the NRA is the oldest civil rights organization in America?",
"Our 2nd amendment right separates us from every country out there. It was derived out of tyranny not hunting or target shooting. It was the 2nd ammendment after free speech. It was that important.",
"I get the reason some people want gun control.  I want them to understand that where I live, we need guns. We are over run with coyotes. They kill our pets. We try and kill them. You have to use a weapon with range. What youd call an assault weapon is perfect for that, and they are popular here for coyotes. I just scare them with my gun, because I don't ever hit them",
"A little history will do you alot of good. Facts of history are not something someone made up, they are facts. Just like the fact vaccines help us fight disease, so a good guy trained with a gun is better than an unprepared bad guy.  With proper training guns are safer than you driving while texting.",
"The 2nd amendment is to ward off tyrannical governments. What does China, North Korea and Nazi Germany have in common. Citizens couldn't and still can't defend themselves when their governments get out of hand. I'd rather have the 2A then not.",
"I believe in in being cautious, but at what cost? It just seems like everyone wants to start taking our rights away. I hope that we don't lose the right to protect our families and friends.",
"The 2nd Amendmentwas to prevent , from invasion and Tyrrany- They first thing the enemy want is your weapons!",
"I think guns law are already strick enough.",
"No more gun control. Just enforce the laws that are ALREADY in place.",
"The 2nd amendment protects my rights and shall not be infringed upon",
"Chicago has some of the strictest gun control laws. That didn't stop the the rising gun related killings.",
"There is no benefit to gun control"
]

In [10]:
lib = []

for p in lib_posts:
    ids = list(all_posts[all_posts.postOrCommentText.apply(lambda x: x == p)].postOrCommentId)
    if len(ids) == 0:
        print(p, '\n')
    lib += ids

lib

['654e2ecbf18dbd001f7aaef2',
 '655d0ae8f1c1b5001f5cb571',
 '655d0ae8f1c1b5001f5cb413',
 '654e2ecbf18dbd001f7aaef2',
 '655d0ae8f1c1b5001f5cb413',
 '654e2ecbf18dbd001f7aaef2',
 '654e2ecbf18dbd001f7aaef2',
 '655d0ae8f1c1b5001f5cb571',
 '654e2ecbf18dbd001f7aaef2',
 '654e2ecbf18dbd001f7aaef2',
 '655d0ae8f1c1b5001f5cb571',
 '655d0ae8f1c1b5001f5cb571',
 '655d0ae8f1c1b5001f5cb571',
 '655d0ae8f1c1b5001f5cb571',
 '654e2ecbf18dbd001f7aaef2',
 '655d0ae8f1c1b5001f5cb413',
 '654e2ecbf18dbd001f7aaef2',
 '654e2ecbf18dbd001f7aaef2',
 '654e2ecbf18dbd001f7aaef2',
 '655d0ae8f1c1b5001f5cb413',
 '654e2ecbf18dbd001f7aaef2',
 '655d0ae8f1c1b5001f5cb571',
 '654e2ecbf18dbd001f7aaef2',
 '654e2ecbf18dbd001f7aaef2',
 '655d0ae8f1c1b5001f5cb413',
 '654e2ecbf18dbd001f7aaef2',
 '655d0ae8f1c1b5001f5cb413',
 '655d0ae8f1c1b5001f5cb413',
 '655d0ae8f1c1b5001f5cb571',
 '655d0ae8f1c1b5001f5cb413',
 '654e2ecbf18dbd001f7aaef2',
 '655d0ae8f1c1b5001f5cb571',
 '655d0ae8f1c1b5001f5cb413',
 '654e2ecbf18dbd001f7aaef2',
 '654e2ecbf18d

In [11]:
con = []

for p in con_posts:
    ids = list(all_posts[all_posts.postOrCommentText.apply(lambda x: x == p)].postOrCommentId)
    if len(ids) == 0:
        print(p, '\n')
    con += ids

con

['654e2eccf18dbd001f7ab25b',
 '655d0ae8f1c1b5001f5cb590',
 '655d0ae8f1c1b5001f5cb555',
 '654e2eccf18dbd001f7ab25b',
 '655d0ae8f1c1b5001f5cb555',
 '654e2eccf18dbd001f7ab25b',
 '654e2eccf18dbd001f7ab25b',
 '655d0ae8f1c1b5001f5cb590',
 '654e2eccf18dbd001f7ab25b',
 '654e2eccf18dbd001f7ab25b',
 '655d0ae8f1c1b5001f5cb590',
 '655d0ae8f1c1b5001f5cb590',
 '655d0ae8f1c1b5001f5cb590',
 '655d0ae8f1c1b5001f5cb590',
 '654e2eccf18dbd001f7ab25b',
 '655d0ae8f1c1b5001f5cb555',
 '654e2eccf18dbd001f7ab25b',
 '654e2eccf18dbd001f7ab25b',
 '654e2eccf18dbd001f7ab25b',
 '655d0ae8f1c1b5001f5cb555',
 '654e2eccf18dbd001f7ab25b',
 '655d0ae8f1c1b5001f5cb590',
 '654e2eccf18dbd001f7ab25b',
 '654e2eccf18dbd001f7ab25b',
 '655d0ae8f1c1b5001f5cb555',
 '654e2eccf18dbd001f7ab25b',
 '655d0ae8f1c1b5001f5cb555',
 '655d0ae8f1c1b5001f5cb555',
 '655d0ae8f1c1b5001f5cb590',
 '655d0ae8f1c1b5001f5cb555',
 '654e2eccf18dbd001f7ab25b',
 '655d0ae8f1c1b5001f5cb590',
 '655d0ae8f1c1b5001f5cb555',
 '654e2eccf18dbd001f7ab25b',
 '654e2eccf18d

# Get user-level measures

In [12]:
df = post_info[post_info.userContent == 1]
df

,sessionId,treatmentArm,participantCode,profilePicture,firstName,lastName,bio,postOrCommentId,userContent,staticContent,...,INSULT_EXPERIMENTAL,PROFANITY_EXPERIMENTAL,THREAT_EXPERIMENTAL,SEXUALLY_EXPLICIT,FLIRTATION,perspective_toxicity_agg,ih_austin_a,ih_austin_b,tox_austin_a,tox_austin_b
0,657895d16d158d001f181302,imm_pc,zf1er9,https://sma2-prod.s3.amazonaws.com/animal%2C%2...,Ginger,NaN,I love cats!,657897906d158d001f182280,1,0,...,0.127028,0.077732,0.010784,0.038217,0.340470,0.128365,0.469136,0.438154,0.254629,0.120118
1,657895d16d158d001f181302,imm_pc,zf1er9,https://sma2-prod.s3.amazonaws.com/animal%2C%2...,Ginger,NaN,I love cats!,657899146d158d001f1856cc,1,0,...,0.033460,0.017778,0.006965,0.006812,0.556122,0.110948,0.351996,0.417278,0.150481,0.061872
2,657895d16d158d001f181302,imm_pc,zf1er9,https://sma2-prod.s3.amazonaws.com/animal%2C%2...,Ginger,NaN,I love cats!,657899606d158d001f1862d5,1,0,...,0.007513,0.011714,0.006848,0.006193,0.241154,0.035897,0.459829,0.369664,0.010807,0.006681
3,657896b96d158d001f181516,imm_treat,8gch37,https://sma2-prod.s3.amazonaws.com/animal%2C%2...,Yelena22,NaN,NC girl,657898246d158d001f18395d,1,0,...,0.248355,0.045670,0.006965,0.009672,0.299508,0.150781,0.342446,0.345526,0.309935,0.170582
4,657896b96d158d001f181516,imm_treat,8gch37,https://sma2-prod.s3.amazonaws.com/animal%2C%2...,Yelena22,NaN,NC girl,657899456d158d001f185f72,1,0,...,0.007969,0.010502,0.006045,0.003052,0.224278,0.033593,0.382053,0.364194,0.012817,0.006676
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4389,657b61122855e7001f93f0ef,imm_control3,pqdpcp,https://sma2-prod.s3.amazonaws.com/animal%2C%2...,Cbros,NaN,Baba,657b62628b12dc001f05d62b,1,0,...,0.027089,0.019554,0.008117,0.004954,0.225375,0.067105,0.128559,0.156050,0.142560,0.057711
4390,657b64a78b12dc001f06215b,imm_control3,h63zav,https://sma2-prod.s3.amazonaws.com/animal%2C%2...,Shan,NaN,"Jersey girl, born and raised!!! I love animals...",657b65a88b12dc001f069f70,1,0,...,0.035223,0.032894,0.033732,0.014331,0.453156,0.096431,0.093467,0.090807,0.172851,0.056027
4391,657b64a78b12dc001f06215b,imm_control3,h63zav,https://sma2-prod.s3.amazonaws.com/animal%2C%2...,Shan,NaN,"Jersey girl, born and raised!!! I love animals...",657b66162855e7001f9554e3,1,0,...,0.012605,0.015147,0.007806,0.008434,0.351283,0.055717,0.548232,0.509349,0.032156,0.015593
4392,657b64a78b12dc001f06215b,imm_control3,h63zav,https://sma2-prod.s3.amazonaws.com/animal%2C%2...,Shan,NaN,"Jersey girl, born and raised!!! I love animals...",657b66cb8b12dc001f07eca7,1,0,...,0.012377,0.010314,0.006654,0.003155,0.388082,0.056238,0.284360,0.376801,0.024496,0.010128


# Aggregate to User-level Measures

In [13]:
# Aggregate all posts

agg_all = df.groupby('participantCode').agg(
    
    # IH A
    ih_a_all = pd.NamedAgg(column="ih_austin_a", aggfunc="mean"),
    
    # IH B
    ih_b_all = pd.NamedAgg(column="ih_austin_b", aggfunc="mean"),

    #Toxicitiy A
    tox_a_all = pd.NamedAgg(column="tox_austin_a", aggfunc="mean"),

    # Toxicity B
    tox_b_all = pd.NamedAgg(column="tox_austin_b", aggfunc="mean")
).reset_index()

agg_all

,participantCode,ih_a_all,ih_b_all,tox_a_all,tox_b_all
0,114n66,0.266236,0.296404,0.048334,0.019166
1,11b468,0.252210,0.323695,0.195428,0.119362
2,11frdz,0.260360,0.285371,0.108976,0.053299
3,11g3d4,0.085161,0.100888,0.028385,0.011247
4,11uby7,0.252306,0.320688,0.093492,0.035447
...,...,...,...,...,...
1440,zxyz1h,0.254910,0.228662,0.166437,0.072189
1441,zy6tqc,0.368552,0.346448,0.055111,0.023966
1442,zywuwt,0.211234,0.208270,0.107075,0.052506
1443,zzmpme,0.251089,0.288543,0.110680,0.044659


In [14]:
# Aggregate all original posts

df_op = df[pd.isnull(df.parentPost)]

agg_op = df_op.groupby('participantCode').agg(
    
    # IH A
    ih_a_op = pd.NamedAgg(column="ih_austin_a", aggfunc="mean"),
    
    # IH B
    ih_b_op = pd.NamedAgg(column="ih_austin_b", aggfunc="mean"),

    #Toxicitiy A
    tox_a_op = pd.NamedAgg(column="tox_austin_a", aggfunc="mean"),

    # Toxicity B
    tox_b_op = pd.NamedAgg(column="tox_austin_b", aggfunc="mean")
    
).reset_index()

agg_op

,participantCode,ih_a_op,ih_b_op,tox_a_op,tox_b_op
0,114n66,0.343366,0.384123,0.078460,0.030371
1,11b468,0.364973,0.453152,0.063027,0.024358
2,11frdz,0.522249,0.582540,0.051069,0.023632
3,11g3d4,0.085161,0.100888,0.028385,0.011247
4,11uby7,0.313038,0.375686,0.084791,0.033744
...,...,...,...,...,...
1290,zxyz1h,0.183477,0.187207,0.237751,0.112069
1291,zy6tqc,0.350682,0.349470,0.054770,0.022682
1292,zywuwt,0.213981,0.240642,0.026028,0.011414
1293,zzmpme,0.251089,0.288543,0.110680,0.044659


In [17]:
# Aggregate all comments

df_comm = df[pd.notnull(df.parentPost)]

agg_comm = df_comm.groupby('participantCode').agg(
    
    # IH A
    ih_a_comm = pd.NamedAgg(column="ih_austin_a", aggfunc="mean"),
    
    # IH B
    ih_b_comm = pd.NamedAgg(column="ih_austin_b", aggfunc="mean"),

    #Toxicitiy A
    tox_a_comm = pd.NamedAgg(column="tox_austin_a", aggfunc="mean"),

    # Toxicity B
    tox_b_comm = pd.NamedAgg(column="tox_austin_b", aggfunc="mean"),

    # Number of Comments
    num_messages_comm =pd.NamedAgg(column="tox_austin_a", aggfunc="size")
    
).reset_index()

agg_comm

,participantCode,ih_a_comm,ih_b_comm,tox_a_comm,tox_b_comm,num_messages_comm
0,114n66,0.256594,0.285439,0.044568,0.017766,8
1,11b468,0.195829,0.258967,0.261628,0.166864,2
2,11frdz,0.173064,0.186315,0.128279,0.063189,3
3,11uby7,0.191574,0.265690,0.102193,0.037150,1
4,12ad6y,0.396042,0.347071,0.021909,0.011133,2
...,...,...,...,...,...,...
1261,zx37ux,0.333937,0.323493,0.108110,0.055151,3
1262,zxyz1h,0.271394,0.238228,0.149980,0.062987,13
1263,zy6tqc,0.376210,0.345153,0.055256,0.024516,7
1264,zywuwt,0.209861,0.192083,0.147598,0.073052,2


In [20]:
# Aggregate all comments on IH posts

df_ih = df[[i in ih for i in df.parentPost]]

agg_ih = df_ih.groupby('participantCode').agg(
    
    # IH A
    ih_a_ih = pd.NamedAgg(column="ih_austin_a", aggfunc="mean"),
    
    # IH B
    ih_b_ih = pd.NamedAgg(column="ih_austin_b", aggfunc="mean"),

    #Toxicitiy A
    tox_a_ih = pd.NamedAgg(column="tox_austin_a", aggfunc="mean"),

    # Toxicity B
    tox_b_ih = pd.NamedAgg(column="tox_austin_b", aggfunc="mean"),

    # Number of IH Comments
    num_messages_ih =pd.NamedAgg(column="tox_austin_a", aggfunc="size")
    
).reset_index()

agg_ih

,participantCode,ih_a_ih,ih_b_ih,tox_a_ih,tox_b_ih,num_messages_ih
0,11frdz,0.173064,0.186315,0.128279,0.063189,3
1,17xpyv,0.186691,0.240587,0.093515,0.036837,1
2,1bqxnq,0.485825,0.395289,0.010116,0.006244,1
3,1c87zz,0.620381,0.611986,0.008168,0.005664,1
4,1f4cwc,0.437144,0.467190,0.172625,0.093962,2
...,...,...,...,...,...,...
288,zu9wre,0.479444,0.392881,0.006220,0.004736,1
289,zvf1ru,0.475079,0.387230,0.011498,0.006916,1
290,zwphpx,0.161204,0.174138,0.041157,0.016532,5
291,zy6tqc,0.259986,0.316250,0.060258,0.024493,1


In [21]:
# Aggregate all comments on Lib posts

df_lib = df[[i in lib for i in df.parentPost]]

agg_lib = df_lib.groupby('participantCode').agg(
    
    # IH A
    ih_a_lib = pd.NamedAgg(column="ih_austin_a", aggfunc="mean"),
    
    # IH B
    ih_b_lib = pd.NamedAgg(column="ih_austin_b", aggfunc="mean"),

    #Toxicitiy A
    tox_a_lib = pd.NamedAgg(column="tox_austin_a", aggfunc="mean"),

    # Toxicity B
    tox_b_lib = pd.NamedAgg(column="tox_austin_b", aggfunc="mean")
    
).reset_index()

agg_lib

,participantCode,ih_a_lib,ih_b_lib,tox_a_lib,tox_b_lib
0,11b468,0.250499,0.330831,0.052058,0.021877
1,11frdz,0.130030,0.127749,0.322361,0.161132
2,12cpx2,0.165415,0.162022,0.106985,0.035733
3,16rhru,0.305004,0.306038,0.095677,0.034662
4,1art8b,0.172380,0.216609,0.031213,0.012253
...,...,...,...,...,...
465,zwcmc7,0.445385,0.367337,0.010179,0.007561
466,zwphpx,0.487332,0.395040,0.007571,0.005732
467,zxyz1h,0.394786,0.323643,0.016147,0.010115
468,zy6tqc,0.480017,0.420835,0.012817,0.006992


In [22]:
# Aggregate all comments on Con posts

df_con = df[[i in con for i in df.parentPost]]

agg_con = df_con.groupby('participantCode').agg(
    
    # IH A
    ih_a_con = pd.NamedAgg(column="ih_austin_a", aggfunc="mean"),
    
    # IH B
    ih_b_con = pd.NamedAgg(column="ih_austin_b", aggfunc="mean"),

    #Toxicitiy A
    tox_a_con = pd.NamedAgg(column="tox_austin_a", aggfunc="mean"),

    # Toxicity B
    tox_b_con = pd.NamedAgg(column="tox_austin_b", aggfunc="mean")
    
).reset_index()

agg_con

,participantCode,ih_a_con,ih_b_con,tox_a_con,tox_b_con
0,11b468,0.141159,0.187104,0.471199,0.311851
1,11frdz,0.194581,0.215597,0.031238,0.014217
2,11uby7,0.191574,0.265690,0.102193,0.037150
3,12cpx2,0.393745,0.351364,0.111618,0.050148
4,17xpyv,0.186691,0.240587,0.093515,0.036837
...,...,...,...,...,...
500,zv3m1r,0.184460,0.190587,0.157564,0.072751
501,zwphpx,0.160355,0.172953,0.041902,0.016833
502,zxyz1h,0.138838,0.130494,0.332537,0.135658
503,zy6tqc,0.259986,0.316250,0.060258,0.024493


In [23]:
# Combine the aggregated data

content_df = agg_all

for agg in [agg_op, agg_comm, agg_ih, agg_con, agg_lib]:
    content_df = content_df.merge(agg, on='participantCode', how="left")

content_df = content_df

content_df
    

,participantCode,ih_a_all,ih_b_all,tox_a_all,tox_b_all,ih_a_op,ih_b_op,tox_a_op,tox_b_op,ih_a_comm,...,tox_b_ih,num_messages_ih,ih_a_con,ih_b_con,tox_a_con,tox_b_con,ih_a_lib,ih_b_lib,tox_a_lib,tox_b_lib
0,114n66,0.266236,0.296404,0.048334,0.019166,0.343366,0.384123,0.078460,0.030371,0.256594,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,11b468,0.252210,0.323695,0.195428,0.119362,0.364973,0.453152,0.063027,0.024358,0.195829,...,NaN,NaN,0.141159,0.187104,0.471199,0.311851,0.250499,0.330831,0.052058,0.021877
2,11frdz,0.260360,0.285371,0.108976,0.053299,0.522249,0.582540,0.051069,0.023632,0.173064,...,0.063189,3.0,0.194581,0.215597,0.031238,0.014217,0.130030,0.127749,0.322361,0.161132
3,11g3d4,0.085161,0.100888,0.028385,0.011247,0.085161,0.100888,0.028385,0.011247,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,11uby7,0.252306,0.320688,0.093492,0.035447,0.313038,0.375686,0.084791,0.033744,0.191574,...,NaN,NaN,0.191574,0.265690,0.102193,0.037150,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1440,zxyz1h,0.254910,0.228662,0.166437,0.072189,0.183477,0.187207,0.237751,0.112069,0.271394,...,NaN,NaN,0.138838,0.130494,0.332537,0.135658,0.394786,0.323643,0.016147,0.010115
1441,zy6tqc,0.368552,0.346448,0.055111,0.023966,0.350682,0.349470,0.054770,0.022682,0.376210,...,0.024493,1.0,0.259986,0.316250,0.060258,0.024493,0.480017,0.420835,0.012817,0.006992
1442,zywuwt,0.211234,0.208270,0.107075,0.052506,0.213981,0.240642,0.026028,0.011414,0.209861,...,NaN,NaN,NaN,NaN,NaN,NaN,0.129048,0.118807,0.282002,0.138659
1443,zzmpme,0.251089,0.288543,0.110680,0.044659,0.251089,0.288543,0.110680,0.044659,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


# Merge with Participant Data

In [24]:
user_df = pd.read_csv("Cleaned_study_data.csv")

In [25]:
posit = {'Strongly agree': "lib",
         "Agree": "lib",
          'Somewhat agree': "lib",
          'Neither agree nor disagree': np.nan,
          'Somewhat disagree': "con",
         "Disagree": "con",
          'Strongly disagree': "con"}

def posit_func(x):
    if x.topic == "gun control":
        return posit[x.pre_policy_gc]
    elif x.topic == "immigration":
        return posit[x.pre_policy_imm]

user_df['posit'] = user_df.apply(posit_func,axis=1)

user_df[['posit','pre_policy_imm','pre_policy_gc',"topic"]]

,posit,pre_policy_imm,pre_policy_gc,topic
0,lib,Agree,Strongly agree,immigration
1,con,Strongly agree,Strongly disagree,gun control
2,con,Strongly disagree,Strongly agree,immigration
3,lib,Somewhat agree,Strongly agree,gun control
4,con,Disagree,Somewhat agree,immigration
...,...,...,...,...
1038,lib,Somewhat agree,Agree,immigration
1039,lib,Strongly agree,Strongly agree,immigration
1040,NaN,Neither agree nor disagree,Agree,immigration
1041,lib,Disagree,Strongly agree,gun control


In [26]:
len(set(user_df.participant_code))

1043

In [27]:
new_df = content_df.merge(user_df, left_on="participantCode", right_on="participant_code", how="right", validate="1:1")
new_df = new_df.drop(columns=['participantCode'])
new_df

,ih_a_all,ih_b_all,tox_a_all,tox_b_all,ih_a_op,ih_b_op,tox_a_op,tox_b_op,ih_a_comm,ih_b_comm,...,mc_correct,n_op,n_comments,zip_state_match,speeder,commented,posted,passcode,gold_star,posit
0,0.361823,0.348515,0.133737,0.070089,0.360970,0.335827,0.078460,0.033009,0.362249,0.354860,...,1,1,2,1,0,1,1,True,0,lib
1,0.362019,0.410255,0.085755,0.031098,0.512410,0.554325,0.057006,0.020452,0.211628,0.266185,...,1,1,1,1,1,1,1,True,1,con
2,0.383066,0.322223,0.009911,0.006789,0.267132,0.233647,0.013194,0.008441,0.499001,0.410800,...,1,1,1,1,1,1,1,True,1,con
3,0.282872,0.349636,0.096769,0.038011,0.379054,0.458685,0.100023,0.039186,0.186691,0.240587,...,1,1,1,1,1,1,1,True,1,lib
4,0.425425,0.437438,0.051341,0.021750,0.256750,0.272360,0.091122,0.035728,0.594101,0.602517,...,1,1,1,1,1,1,1,True,1,con
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1038,0.235552,0.223638,0.035396,0.015886,0.220500,0.228770,0.051316,0.018822,0.238562,0.222612,...,1,1,5,1,1,1,1,True,1,lib
1039,0.383666,0.378626,0.028399,0.013212,NaN,NaN,NaN,NaN,0.383666,0.378626,...,1,0,4,1,1,0,1,True,0,lib
1040,0.177561,0.166600,0.024242,0.011742,0.106842,0.112467,0.030388,0.013307,0.248280,0.220734,...,1,1,1,1,1,1,1,True,1,NaN
1041,0.395352,0.387153,0.038950,0.016728,0.299257,0.251120,0.096552,0.039512,0.414571,0.414359,...,1,1,5,1,1,1,1,True,1,lib


In [28]:
len(set(new_df.participant_code))

1043

In [29]:
data = []

for ind, row in new_df.iterrows():
    datum = {"participantCode": row['participant_code']}

    if row.posit == "lib":

        # IH A
        datum['ih_a_ca'] = row.ih_a_con
        
        # IH B
        datum['ih_b_ca'] = row.ih_b_con
    
        #Toxicitiy A
        datum['tox_a_ca'] = row.tox_a_con
    
        # Toxicity B
        datum['tox_b_ca'] = row.tox_b_con

    elif row.posit == "con":
        
        # IH A
        datum['ih_a_ca'] = row.ih_a_lib
        
        # IH B
        datum['ih_b_ca'] = row.ih_b_lib
    
        #Toxicitiy A
        datum['tox_a_ca'] = row.tox_a_lib
    
        # Toxicity B
        datum['tox_b_ca'] = row.tox_b_lib

    data.append(datum)

ca_df = pd.DataFrame(data)
ca_df

,participantCode,ih_a_ca,ih_b_ca,tox_a_ca,tox_b_ca
0,8gch37,0.382053,0.364194,0.012817,0.006676
1,nyuhem,NaN,NaN,NaN,NaN
2,gw6r7f,NaN,NaN,NaN,NaN
3,17xpyv,0.186691,0.240587,0.093515,0.036837
4,hfyv7m,NaN,NaN,NaN,NaN
...,...,...,...,...,...
1038,r9hyrv,0.244652,0.229313,0.035145,0.015861
1039,3naupc,0.343404,0.298406,0.010430,0.006796
1040,6fxpk1,NaN,NaN,NaN,NaN
1041,pwghnt,0.192040,0.181048,0.027324,0.015055


In [30]:
len(set(new_df.participant_code))

1043

In [31]:
new_df = new_df.merge(ca_df, left_on = "participant_code", right_on="participantCode", how="left", validate="1:1").drop(columns="participantCode")

In [32]:
new_df

,ih_a_all,ih_b_all,tox_a_all,tox_b_all,ih_a_op,ih_b_op,tox_a_op,tox_b_op,ih_a_comm,ih_b_comm,...,speeder,commented,posted,passcode,gold_star,posit,ih_a_ca,ih_b_ca,tox_a_ca,tox_b_ca
0,0.361823,0.348515,0.133737,0.070089,0.360970,0.335827,0.078460,0.033009,0.362249,0.354860,...,0,1,1,True,0,lib,0.382053,0.364194,0.012817,0.006676
1,0.362019,0.410255,0.085755,0.031098,0.512410,0.554325,0.057006,0.020452,0.211628,0.266185,...,1,1,1,True,1,con,NaN,NaN,NaN,NaN
2,0.383066,0.322223,0.009911,0.006789,0.267132,0.233647,0.013194,0.008441,0.499001,0.410800,...,1,1,1,True,1,con,NaN,NaN,NaN,NaN
3,0.282872,0.349636,0.096769,0.038011,0.379054,0.458685,0.100023,0.039186,0.186691,0.240587,...,1,1,1,True,1,lib,0.186691,0.240587,0.093515,0.036837
4,0.425425,0.437438,0.051341,0.021750,0.256750,0.272360,0.091122,0.035728,0.594101,0.602517,...,1,1,1,True,1,con,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1038,0.235552,0.223638,0.035396,0.015886,0.220500,0.228770,0.051316,0.018822,0.238562,0.222612,...,1,1,1,True,1,lib,0.244652,0.229313,0.035145,0.015861
1039,0.383666,0.378626,0.028399,0.013212,NaN,NaN,NaN,NaN,0.383666,0.378626,...,1,0,1,True,0,lib,0.343404,0.298406,0.010430,0.006796
1040,0.177561,0.166600,0.024242,0.011742,0.106842,0.112467,0.030388,0.013307,0.248280,0.220734,...,1,1,1,True,1,NaN,NaN,NaN,NaN,NaN
1041,0.395352,0.387153,0.038950,0.016728,0.299257,0.251120,0.096552,0.039512,0.414571,0.414359,...,1,1,1,True,1,lib,0.192040,0.181048,0.027324,0.015055


In [33]:
len(set(new_df.participant_code))

1043

In [35]:
new_df.to_csv("PerspectiveOutcomes.csv", index=False)

# Let the Mining Begin!

In [37]:
new_df.columns.to_list()

['participantCode',
 'ih_a_all',
 'ih_b_all',
 'tox_a_all',
 'tox_b_all',
 'ih_a_op',
 'ih_b_op',
 'tox_a_op',
 'tox_b_op',
 'ih_a_comm',
 'ih_b_comm',
 'tox_a_comm',
 'tox_b_comm',
 'ih_a_ih',
 'ih_b_ih',
 'tox_a_ih',
 'tox_b_ih',
 'ih_a_con',
 'ih_b_con',
 'tox_a_con',
 'tox_b_con',
 'ih_a_lib',
 'ih_b_lib',
 'tox_a_lib',
 'tox_b_lib',
 'StartDate_post',
 'EndDate_post',
 'Status_post',
 'IPAddress_post',
 'Progress_post',
 'Duration (in seconds)_post',
 'Finished_post',
 'RecordedDate_post',
 'ResponseId_post',
 'RecipientLastName_post',
 'RecipientFirstName_post',
 'RecipientEmail_post',
 'ExternalReference_post',
 'LocationLatitude_post',
 'LocationLongitude_post',
 'DistributionChannel_post',
 'UserLanguage_post',
 'timer.panas_First Click',
 'timer.panas_Last Click',
 'timer.panas_Page Submit',
 'timer.panas_Click Count',
 'panas_active',
 'panas_attentive',
 'panas_alert',
 'panas_determined',
 'panas_inspired',
 'panas_hostile',
 'panas_ashamed',
 'panas_upset',
 'panas_afraid

In [38]:
ys = list(new_df.columns)[1:25] + list(new_df.columns)[-4:]

for y in ys:
    print(y, ": ", stats.ttest_ind(new_df[new_df.condition == "treat"][y],
                                   new_df[new_df.condition == "control"][y],
                                   nan_policy="omit"))

ih_a_all :  Ttest_indResult(statistic=2.0428747340981777, pvalue=0.04133222593673377)
ih_b_all :  Ttest_indResult(statistic=2.0395140047792326, pvalue=0.04166682328167753)
tox_a_all :  Ttest_indResult(statistic=0.9931792781827891, pvalue=0.3208680019837121)
tox_b_all :  Ttest_indResult(statistic=0.9113809948129217, pvalue=0.36231901323527327)
ih_a_op :  Ttest_indResult(statistic=2.6476232430026703, pvalue=0.008251477465826445)
ih_b_op :  Ttest_indResult(statistic=2.1919595444939692, pvalue=0.028644655104448878)
tox_a_op :  Ttest_indResult(statistic=0.8326744644871502, pvalue=0.4052550415053088)
tox_b_op :  Ttest_indResult(statistic=0.8935443716607412, pvalue=0.37181085276290793)
ih_a_comm :  Ttest_indResult(statistic=1.0760915437358418, pvalue=0.28218531933341195)
ih_b_comm :  Ttest_indResult(statistic=1.1488456385982624, pvalue=0.25093611671529886)
tox_a_comm :  Ttest_indResult(statistic=1.19591240080014, pvalue=0.23205733533524686)
tox_b_comm :  Ttest_indResult(statistic=0.9503955701

In [39]:
for y in ys:
    r = stats.mannwhitneyu(new_df[new_df.condition == "treat"][y],
                           new_df[new_df.condition == "control"][y],
                           nan_policy="omit")
    if r[1] < 0.05:
        print(y, r)

ih_a_all MannwhitneyuResult(statistic=129001.0, pvalue=0.04326904542276676)
ih_b_all MannwhitneyuResult(statistic=129535.0, pvalue=0.032227463951556165)
ih_a_op MannwhitneyuResult(statistic=106192.0, pvalue=0.008349993871546618)
ih_b_op MannwhitneyuResult(statistic=104436.0, pvalue=0.029985235156663417)


In [40]:
for y in ys:
    r = stats.ttest_ind(new_df[new_df.condition == "treat"][y],
                        new_df[new_df.condition == "control"][y],
                        nan_policy="omit")
    if r[1] < 0.05:
        print(y, r)

ih_a_all Ttest_indResult(statistic=2.0428747340981777, pvalue=0.04133222593673377)
ih_b_all Ttest_indResult(statistic=2.0395140047792326, pvalue=0.04166682328167753)
ih_a_op Ttest_indResult(statistic=2.6476232430026703, pvalue=0.008251477465826445)
ih_b_op Ttest_indResult(statistic=2.1919595444939692, pvalue=0.028644655104448878)


In [41]:
data = []
for y in ys:
    x1 = new_df[new_df.condition == "treat"][y]
    x2 = new_df[new_df.condition == "control"][y]

    d = (x1.mean() - x2.mean()) / new_df[y].std()
    
    u = stats.mannwhitneyu(x1,x2,nan_policy="omit")
    t = stats.ttest_ind(x1,x2,nan_policy="omit")
    
    data.append({"U": u[0],
                 "U_p": u[1],
                 "t": t[0],
                 "t_p": t[1],
                 "y": y,
                 "mu_control": x2.mean(),
                 "mu_treat": x1.mean(),
                 "eta_control": x2.median(),
                 "eta_treat": x1.median(),
                 "d": d})

effects_df = pd.DataFrame(data)
effects_df

,U,U_p,t,t_p,y,mu_control,mu_treat,eta_control,eta_treat,d
0,129001.0,0.043269,2.042875,0.041332,ih_a_all,0.319104,0.332514,0.314348,0.325050,0.130374
1,129535.0,0.032227,2.039514,0.041667,ih_b_all,0.323467,0.337218,0.312011,0.329134,0.130161
2,126715.0,0.132230,0.993179,0.320868,tox_a_all,0.081807,0.086362,0.058698,0.065192,0.063487
3,126651.0,0.135981,0.911381,0.362319,tox_b_all,0.037428,0.039555,0.024072,0.028153,0.058263
4,106192.0,0.008350,2.647623,0.008251,ih_a_op,0.286114,0.308898,0.266263,0.294537,0.178273
5,104436.0,0.029985,2.191960,0.028645,ih_b_op,0.325588,0.346420,0.306997,0.331816,0.147776
6,98096.5,0.628909,0.832674,0.405255,tox_a_op,0.102431,0.108153,0.065797,0.064610,0.056268
7,98265.0,0.597429,0.893544,0.371811,tox_b_op,0.046395,0.049673,0.026709,0.026854,0.060378
8,98220.0,0.295508,1.076092,0.282185,ih_a_comm,0.347354,0.356303,0.347937,0.354113,0.073061
9,99342.0,0.177244,1.148846,0.250936,ih_b_comm,0.328301,0.337288,0.319651,0.329327,0.077994


In [42]:
effects_df[['y','d','t_p']]

,y,d,t_p
0,ih_a_all,0.130374,0.041332
1,ih_b_all,0.130161,0.041667
2,tox_a_all,0.063487,0.320868
3,tox_b_all,0.058263,0.362319
4,ih_a_op,0.178273,0.008251
5,ih_b_op,0.147776,0.028645
6,tox_a_op,0.056268,0.405255
7,tox_b_op,0.060378,0.371811
8,ih_a_comm,0.073061,0.282185
9,ih_b_comm,0.077994,0.250936


In [43]:
effects_df[(effects_df.t_p < 0.05)|(effects_df.U_p < 0.05)].sort_values('y')

,U,U_p,t,t_p,y,mu_control,mu_treat,eta_control,eta_treat,d
0,129001.0,0.043269,2.042875,0.041332,ih_a_all,0.319104,0.332514,0.314348,0.325050,0.130374
4,106192.0,0.008350,2.647623,0.008251,ih_a_op,0.286114,0.308898,0.266263,0.294537,0.178273
1,129535.0,0.032227,2.039514,0.041667,ih_b_all,0.323467,0.337218,0.312011,0.329134,0.130161
5,104436.0,0.029985,2.191960,0.028645,ih_b_op,0.325588,0.346420,0.306997,0.331816,0.147776


In [44]:
effects_df[(effects_df.t_p < 0.05)&(effects_df.U_p < 0.05)]

,U,U_p,t,t_p,y,mu_control,mu_treat,eta_control,eta_treat,d
0,129001.0,0.043269,2.042875,0.041332,ih_a_all,0.319104,0.332514,0.314348,0.325050,0.130374
1,129535.0,0.032227,2.039514,0.041667,ih_b_all,0.323467,0.337218,0.312011,0.329134,0.130161
4,106192.0,0.008350,2.647623,0.008251,ih_a_op,0.286114,0.308898,0.266263,0.294537,0.178273
5,104436.0,0.029985,2.191960,0.028645,ih_b_op,0.325588,0.346420,0.306997,0.331816,0.147776


# Analysis by Time

## Time Bucket Analysis

In [47]:
post_info

,sessionId,treatmentArm,participantCode,profilePicture,firstName,lastName,bio,postOrCommentId,userContent,staticContent,...,INSULT_EXPERIMENTAL,PROFANITY_EXPERIMENTAL,THREAT_EXPERIMENTAL,SEXUALLY_EXPLICIT,FLIRTATION,perspective_toxicity_agg,ih_austin_a,ih_austin_b,tox_austin_a,tox_austin_b
0,657895d16d158d001f181302,imm_pc,zf1er9,https://sma2-prod.s3.amazonaws.com/animal%2C%2...,Ginger,NaN,I love cats!,657897906d158d001f182280,1,0,...,0.127028,0.077732,0.010784,0.038217,0.340470,0.128365,0.469136,0.438154,0.254629,0.120118
1,657895d16d158d001f181302,imm_pc,zf1er9,https://sma2-prod.s3.amazonaws.com/animal%2C%2...,Ginger,NaN,I love cats!,657899146d158d001f1856cc,1,0,...,0.033460,0.017778,0.006965,0.006812,0.556122,0.110948,0.351996,0.417278,0.150481,0.061872
2,657895d16d158d001f181302,imm_pc,zf1er9,https://sma2-prod.s3.amazonaws.com/animal%2C%2...,Ginger,NaN,I love cats!,657899606d158d001f1862d5,1,0,...,0.007513,0.011714,0.006848,0.006193,0.241154,0.035897,0.459829,0.369664,0.010807,0.006681
3,657896b96d158d001f181516,imm_treat,8gch37,https://sma2-prod.s3.amazonaws.com/animal%2C%2...,Yelena22,NaN,NC girl,657898246d158d001f18395d,1,0,...,0.248355,0.045670,0.006965,0.009672,0.299508,0.150781,0.342446,0.345526,0.309935,0.170582
4,657896b96d158d001f181516,imm_treat,8gch37,https://sma2-prod.s3.amazonaws.com/animal%2C%2...,Yelena22,NaN,NC girl,657899456d158d001f185f72,1,0,...,0.007969,0.010502,0.006045,0.003052,0.224278,0.033593,0.382053,0.364194,0.012817,0.006676
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4389,657b61122855e7001f93f0ef,imm_control3,pqdpcp,https://sma2-prod.s3.amazonaws.com/animal%2C%2...,Cbros,NaN,Baba,657b62628b12dc001f05d62b,1,0,...,0.027089,0.019554,0.008117,0.004954,0.225375,0.067105,0.128559,0.156050,0.142560,0.057711
4390,657b64a78b12dc001f06215b,imm_control3,h63zav,https://sma2-prod.s3.amazonaws.com/animal%2C%2...,Shan,NaN,"Jersey girl, born and raised!!! I love animals...",657b65a88b12dc001f069f70,1,0,...,0.035223,0.032894,0.033732,0.014331,0.453156,0.096431,0.093467,0.090807,0.172851,0.056027
4391,657b64a78b12dc001f06215b,imm_control3,h63zav,https://sma2-prod.s3.amazonaws.com/animal%2C%2...,Shan,NaN,"Jersey girl, born and raised!!! I love animals...",657b66162855e7001f9554e3,1,0,...,0.012605,0.015147,0.007806,0.008434,0.351283,0.055717,0.548232,0.509349,0.032156,0.015593
4392,657b64a78b12dc001f06215b,imm_control3,h63zav,https://sma2-prod.s3.amazonaws.com/animal%2C%2...,Shan,NaN,"Jersey girl, born and raised!!! I love animals...",657b66cb8b12dc001f07eca7,1,0,...,0.012377,0.010314,0.006654,0.003155,0.388082,0.056238,0.284360,0.376801,0.024496,0.010128


## Time-based Ablation Analysis